<a href="https://colab.research.google.com/github/iafrincu/MLP-HOG-CIFAR10-classifier/blob/master/Tema_SSPI_Ioana_Frincu_Eduard_Marin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from keras.datasets import cifar10
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

In [2]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

170500096/170498071 [==============================] - 2s 0us/step


In [ ]:
# The skimage defaults (8x8-pixel cells) leave only a 4x4 grid of cells on a 32x32
# image, i.e. 324 features, discarding most spatial detail. 4x4-pixel cells give an
# 8x8 grid and 1,764 features.
# channel_axis=-1 computes gradients on all three colour channels and keeps the
# strongest one at each pixel, so edges between colours of similar brightness
# (lost in grayscale) still count.
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2),
                  block_norm='L2-Hys', channel_axis=-1)

def extract_features(data):
    # run on all CPU cores; each image is independent
    features = Parallel(n_jobs=-1, batch_size=512)(
        delayed(hog)(image, **HOG_PARAMS) for image in data)
    return np.asarray(features, dtype=np.float32)

In [ ]:
# etichetele vin cu forma (n, 1); flatten le aduce la un vector (n,)
y_train = y_train.flatten()
y_test = y_test.flatten()

In [0]:
X_train_features=extract_features(X_train)

In [0]:
X_test_features=extract_features(X_test)

In [0]:
# Hyperparameters are chosen on a validation set held out from the training data.
# The test set is used only once, for the final model; picking the best run by
# its test score would make the reported test accuracy optimistic.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_features, y_train, test_size=0.1, stratify=y_train, random_state=42)

def make_model(hidden, lr):
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=hidden, learning_rate_init=lr,
                      early_stopping=True, n_iter_no_change=5, random_state=42))

# Same 3x3 grid as the original experiments, with learning rates suited to Adam
results = []
for hidden in [(100,), (500,), (1000,)]:
    for lr in [1e-4, 1e-3, 1e-2]:
        start = time.time()
        model = make_model(hidden, lr).fit(X_tr, y_tr)
        results.append({'hidden': hidden, 'lr': lr,
                        'train_acc': model.score(X_tr, y_tr),
                        'val_acc': model.score(X_val, y_val),
                        'epochs': model[-1].n_iter_,
                        'seconds': round(time.time() - start)})
        print(results[-1])

results = pd.DataFrame(results).sort_values('val_acc', ascending=False)
results

In [0]:
# Refit the best configuration on the full training set
best = results.iloc[0]
print(f"Best: hidden={best['hidden']}, lr={best['lr']} (val acc {best['val_acc']:.3f})")
clf = make_model(best['hidden'], best['lr']).fit(X_train_features, y_train)

In [0]:
y_pred = clf.predict(X_test_features)

In [0]:
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}')  # target: at least 0.58
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))

In [ ]:
# Rows are true classes; each row sums to 1, so the diagonal is per-class recall
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASSES,
                                        normalize='true', values_format='.2f',
                                        xticks_rotation=45, cmap='Blues',
                                        colorbar=False, ax=ax)
ax.set_title('Normalised confusion matrix')
plt.tight_layout()
plt.show()